# *Do Accessible Steam Indie Games Get More Player Engagement?* — Data Collection Notebook

**Author:** Palatine Rigaut

**Date:** May, 2026

---

This notebook documents how the raw project data were collected on **2026-05-14**.

It is intentionally guarded by `RUN_COLLECTION = False` so it does not accidentally send many API requests when opened. The already-collected raw files are in `data/raw/`.

In [ ]:
from pathlib import Path
import json
import random
import time
import urllib.request
import datetime

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RUN_COLLECTION = False
SEED = 20260514
TARGET_N = 3000

STEAMSPY_INDIE_URL = 'https://steamspy.com/api.php?request=genre&genre=Indie'
STORE_APPDETAILS_URL = 'https://store.steampowered.com/api/appdetails?appids={appid}&cc=us&l=english'

RAW_STEAMSPY = ROOT / 'data/raw/steamspy_indie_20260514.json'
RAW_STORE = ROOT / 'data/raw/steam_store_appdetails_sample_20260514.jsonl'
SAMPLE_IDS = ROOT / 'data/raw/sample_appids_20260514.json'

## 1. Download SteamSpy Indie List

We used the SteamSpy `genre=Indie` endpoint because it provided a large pre-collected list of indie games together with app IDs, review counts, owner estimates, prices, developer names and publisher information.

In [ ]:
if RUN_COLLECTION:
    req = urllib.request.Request(STEAMSPY_INDIE_URL, headers={'User-Agent': 'school-steam-indie-analysis/1.0'})
    with urllib.request.urlopen(req, timeout=90) as resp:
        steamspy_data = json.loads(resp.read().decode('utf-8', 'replace'))
    RAW_STEAMSPY.write_text(json.dumps(steamspy_data, ensure_ascii=False), encoding='utf-8')
else:
    steamspy_data = json.loads(RAW_STEAMSPY.read_text(encoding='utf-8'))

len(steamspy_data)

## 2. Draw Reproducible Sample

Only apps with at least one positive or negative review were sampled. This avoids rows where `positive_rate` cannot be computed.

In [ ]:
items = list(steamspy_data.values())
for row in items:
    row['total_reviews'] = int(row.get('positive') or 0) + int(row.get('negative') or 0)

eligible = [row for row in items if row['total_reviews'] >= 1]
sample = random.Random(SEED).sample(eligible, min(TARGET_N, len(eligible)))

if RUN_COLLECTION:
    SAMPLE_IDS.write_text(json.dumps({'seed': SEED, 'target_n': len(sample), 'appids': [int(r['appid']) for r in sample]}, indent=2), encoding='utf-8')

len(eligible), len(sample)

## 3. Fetch Store Metadata One App at a Time

Because Steam Store `appdetails` did not support multiple app IDs in a single request during data collection, we queried the API once per app ID with a short delay between requests. During the full collection process, Steam returned HTTP 429 errors after approximately 1,000 successful responses. We therefore retained the successfully collected rows and avoided aggressive retrying.

In [ ]:
if RUN_COLLECTION:
    existing = set()
    if RAW_STORE.exists():
        with open(RAW_STORE, encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    existing.add(int(json.loads(line)['appid']))

    with open(RAW_STORE, 'a', encoding='utf-8') as out:
        for i, row in enumerate(sample, 1):
            appid = int(row['appid'])
            if appid in existing:
                continue
            record = {
                'appid': appid,
                'fetched_at': datetime.datetime.now(datetime.UTC).isoformat(),
                'steamspy': row,
            }
            try:
                req = urllib.request.Request(STORE_APPDETAILS_URL.format(appid=appid), headers={'User-Agent': 'school-steam-indie-analysis/1.0'})
                with urllib.request.urlopen(req, timeout=25) as resp:
                    record['store_response'] = json.loads(resp.read().decode('utf-8', 'replace'))
                record['http_ok'] = True
            except Exception as exc:
                record['http_ok'] = False
                record['error'] = repr(exc)
            out.write(json.dumps(record, ensure_ascii=False) + '
')
            out.flush()
            time.sleep(0.12)
else:
    print('Collection disabled. Set RUN_COLLECTION = True only if you intentionally want to re-query Steam.')

## 4. Check Collected Raw File

In [ ]:
raw_records = [json.loads(line) for line in RAW_STORE.read_text(encoding='utf-8').splitlines() if line.strip()]
http_ok = sum(1 for r in raw_records if r.get('http_ok'))
errors = len(raw_records) - http_ok
print(f'Raw records: {len(raw_records)}')
print(f'HTTP OK: {http_ok}')
print(f'Errors/rate-limited rows: {errors}')